In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.ToTensor()
train_loader = DataLoader(datasets.MNIST('.', train=True, download=True, transform=transform), batch_size=64, shuffle=True)

test_loader = DataLoader(datasets.MNIST('.', train=False, transform=transform), batch_size=1000, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 116MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 26.1MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 62.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.26MB/s]


In [3]:

def evaluate(model, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            output = model(X)
            pred = output.argmax(dim=1)
            correct += pred.eq(y).sum().item()
    return correct / len(test_loader.dataset)


In [4]:

class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_sizes=[128, 64], output_dim=10, activation=nn.ReLU):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(activation())
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


In [5]:

def train_and_evaluate(hidden_layers, activation, optimizer_fn):
    model = MLP(hidden_sizes=hidden_layers, activation=activation).to(device)
    optimizer = optimizer_fn(model.parameters())
    criterion = nn.CrossEntropyLoss()

    start = time.time()
    for epoch in range(5):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
    end = time.time()

    acc = evaluate(model, test_loader)
    return acc, end - start


In [6]:

acc1, time1 = train_and_evaluate([128, 64], nn.ReLU, lambda p: optim.Adam(p))
print("A.1 ReLU + Adam → Acc:", acc1, "Time:", time1)

acc2, time2 = train_and_evaluate([128, 64], nn.Sigmoid, lambda p: optim.Adam(p))
print("A.2 Sigmoid + Adam → Acc:", acc2, "Time:", time2)

acc3, time3 = train_and_evaluate([128, 64], nn.ReLU, lambda p: optim.SGD(p, lr=0.01, momentum=0.9))
print("A.3 ReLU + SGD → Acc:", acc3, "Time:", time3)

acc4, time4 = train_and_evaluate([256, 128, 64, 32], nn.ReLU, lambda p: optim.Adam(p))
print("A.4 5-Layer ReLU + Adam → Acc:", acc4, "Time:", time4)


A.1 ReLU + Adam → Acc: 0.9749 Time: 48.42197608947754
A.2 Sigmoid + Adam → Acc: 0.9682 Time: 47.65228247642517
A.3 ReLU + SGD → Acc: 0.9706 Time: 44.228981494903564
A.4 5-Layer ReLU + Adam → Acc: 0.9741 Time: 63.23931884765625
